# 02 — Capped local dataset download

This laptop-first notebook downloads the complete GLueCoS datasets and a capped CulturaX pretraining corpus to local storage. It streams the first configured number of UTF-8 text bytes for English, Spanish, and Hindi, so it does not download each full CulturaX subset. GLueCoS targets use `Dataset.save_to_disk()`; the large raw CulturaX streams are written directly as Parquet to avoid a second full local copy. Every target is tracked by an atomic local manifest, allowing interrupted downloads to resume from the Hugging Face cache and completed targets to be skipped.

Run `01_environment_setup.ipynb` first. Internet access is required only while downloading.

## Before downloading

This notebook requests all five GLueCoS datasets plus the first 15,000,000,000 UTF-8 text bytes of each English, Spanish, and Hindi CulturaX stream. Ensure the laptop has sufficient local disk space for both the Hugging Face cache and persisted datasets. CulturaX is gated: accept its Hugging Face terms and set `HF_TOKEN` in the local environment before starting. Tokens are never saved to disk by this notebook.

In [1]:
import os
from pathlib import Path

def find_project_root(start_directory):
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / "instructions.md").is_file():
            return candidate.resolve()
    return None

configured_root = os.environ.get("PROJECT_ROOT")
PROJECT_ROOT = (
    Path(configured_root).expanduser().resolve()
    if configured_root else find_project_root(Path.cwd().resolve())
)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "instructions.md").is_file():
    raise FileNotFoundError(
        "Could not find PROJECT_ROOT. Start Jupyter from the repository root or set PROJECT_ROOT before launching it."
    )

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\shenr\tokenization-research


## 1. Load local configuration and checkpoint manifest

The manifest is written atomically before and after every target. A target is considered complete only when the manifest says `completed` and its saved artifact validates (`load_from_disk()` for GLueCoS or a readable Parquet file for CulturaX). Existing partial folders are protected from overwrite.

In [2]:
import json
import importlib.metadata
from datetime import datetime, timezone

import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Missing configs/config.yaml. Run 01_environment_setup.ipynb first.")

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

download_config = config["data_download"]
if download_config.get("mode") != "capped":
    raise ValueError("This notebook requires data_download.mode: capped.")
MAX_TEXT_BYTES_PER_LANGUAGE = int(download_config["max_text_bytes_per_language"])
if MAX_TEXT_BYTES_PER_LANGUAGE <= 0:
    raise ValueError("data_download.max_text_bytes_per_language must be positive.")
PARQUET_SHARD_ROWS = int(download_config["parquet_shard_rows"])
if PARQUET_SHARD_ROWS <= 0:
    raise ValueError("data_download.parquet_shard_rows must be positive.")
if "culturax" not in download_config.get("sources", {}):
    raise KeyError("data_download.sources.culturax is required.")

RAW_DATA_ROOT = PROJECT_ROOT / config["paths"]["raw_data"]
HF_CACHE_ROOT = PROJECT_ROOT / config["paths"]["hf_cache"]
HF_HUB_CACHE_ROOT = HF_CACHE_ROOT / "hub"
HF_DATASETS_CACHE_ROOT = HF_CACHE_ROOT / "datasets"
EXPERIMENT_ROOT = PROJECT_ROOT / config["paths"]["experiments"] / "dataset_download"
for directory in (RAW_DATA_ROOT, HF_HUB_CACHE_ROOT, HF_DATASETS_CACHE_ROOT, EXPERIMENT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

# Keep all Hugging Face Hub and Datasets cache files on the local project disk.
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE_ROOT)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE_ROOT)

import datasets

installed_datasets_version = importlib.metadata.version("datasets")
loaded_datasets_version = datasets.__version__
if loaded_datasets_version != installed_datasets_version:
    raise RuntimeError(
        f"The kernel has datasets=={loaded_datasets_version} loaded, but the environment now has "
        f"datasets=={installed_datasets_version}. Restart the VS Code Jupyter kernel, then run this notebook from the top."
    )
if tuple(map(int, loaded_datasets_version.split(".")[:2])) < (4, 8):
    raise RuntimeError(
        f"datasets=={loaded_datasets_version} is too old for Python 3.14. Run 01_environment_setup.ipynb, "
        "restart the kernel, then rerun this notebook."
    )

MANIFEST_PATH = EXPERIMENT_ROOT / "manifest.json"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    return {"created_at_utc": utc_now(), "targets": {}}

def save_manifest(manifest):
    temporary_path = MANIFEST_PATH.with_suffix(".tmp")
    temporary_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary_path.replace(MANIFEST_PATH)

manifest = load_manifest()
print(f"Manifest: {MANIFEST_PATH}")
print(f"datasets version: {loaded_datasets_version}")
print(f"Hugging Face Hub cache: {HF_HUB_CACHE_ROOT}")
print(f"Hugging Face Datasets cache: {HF_DATASETS_CACHE_ROOT}")
print(f"Download mode: capped ({MAX_TEXT_BYTES_PER_LANGUAGE:,} UTF-8 text bytes per language)")


Manifest: C:\Users\shenr\tokenization-research\experiments\dataset_download\manifest.json
datasets version: 5.0.0
Hugging Face Hub cache: C:\Users\shenr\tokenization-research\data\cache\huggingface\hub
Hugging Face Datasets cache: C:\Users\shenr\tokenization-research\data\cache\huggingface\datasets
Download mode: capped (15,000,000,000 UTF-8 text bytes per language)


## 2. Authenticate for gated data

Set `HF_TOKEN` in the local environment before launching Jupyter if CulturaX access is required. The token remains in memory only. Public datasets can download without it.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    print("Hugging Face token loaded from the local environment and will be passed in memory to dataset loaders.")
else:
    print("No HF_TOKEN was found. CulturaX will fail until its access terms are accepted and HF_TOKEN is set.")


Hugging Face token loaded from the local environment and will be passed in memory to dataset loaders.


## 3. Restart-safe download helpers

The helpers keep completed datasets intact. If a target fails before it is saved, rerun the notebook: Hugging Face will reuse the local cache. If an incomplete `.incomplete` directory exists, inspect it before moving or removing it manually; this notebook will not delete data.

In [4]:
from datasets import load_dataset, load_from_disk

def safe_path_component(value):
    return value.replace("/", "__").replace(" ", "_")

def validate_saved_dataset(destination):
    try:
        dataset = load_from_disk(str(destination))
        if hasattr(dataset, "items"):
            return all(len(split) > 0 for _, split in dataset.items())
        return len(dataset) > 0
    except Exception:
        return False

def target_is_complete(target_id, destination):
    entry = manifest["targets"].get(target_id, {})
    return entry.get("status") == "completed" and validate_saved_dataset(destination)

def save_target(target_id, destination, loader, metadata):
    if target_is_complete(target_id, destination):
        print(f"Checkpoint found; skipping {target_id}: {destination}")
        return manifest["targets"][target_id]

    temporary_destination = destination.with_name(f"{destination.name}.incomplete")
    if destination.exists() or temporary_destination.exists():
        raise FileExistsError(
            f"Found an incomplete or untracked folder for {target_id}: {destination} or {temporary_destination}. "
            "Inspect it before retrying; this notebook will not overwrite existing local data."
        )

    manifest["targets"][target_id] = {
        "status": "running", "started_at_utc": utc_now(),
        "destination": str(destination), "metadata": metadata,
    }
    save_manifest(manifest)

    try:
        dataset = loader()
        destination.parent.mkdir(parents=True, exist_ok=True)
        print(f"Saving dataset to: {temporary_destination}")
        dataset.save_to_disk(str(temporary_destination))
        temporary_destination.replace(destination)
        split_rows = (
            {split_name: len(split) for split_name, split in dataset.items()}
            if hasattr(dataset, "items") else {"dataset": len(dataset)}
        )
        manifest["targets"][target_id].update({
            "status": "completed", "completed_at_utc": utc_now(), "rows": split_rows,
        })
        save_manifest(manifest)
        print(f"Saved {target_id} to: {destination}")
        return manifest["targets"][target_id]
    except Exception as error:
        manifest["targets"][target_id].update({
            "status": "failed", "failed_at_utc": utc_now(),
            "error": f"{type(error).__name__}: {error}",
        })
        save_manifest(manifest)
        raise

def atomic_write_json(path, payload):
    temporary_path = path.with_name(f"{path.name}.tmp")
    temporary_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary_path.replace(path)

def validate_parquet_shard(path):
    try:
        import pyarrow.parquet as pq
        return path.is_file() and pq.ParquetFile(path).metadata.num_rows > 0
    except Exception:
        return False

def validate_saved_parquet(destination):
    try:
        metadata = json.loads((destination / "metadata.json").read_text(encoding="utf-8"))
        parts = metadata.get("parts", [])
        return metadata.get("status") == "completed" and bool(parts) and all(
            validate_parquet_shard(destination / part_name) for part_name in parts
        )
    except Exception:
        return False

def streaming_target_is_complete(target_id, destination):
    entry = manifest["targets"].get(target_id, {})
    return entry.get("status") == "completed" and validate_saved_parquet(destination)

def write_parquet_shards(records, destination, progress, shard_rows, checkpoint):
    import pyarrow as pa
    import pyarrow.parquet as pq

    schema = pa.schema([pa.field("text", pa.string()), pa.field("language", pa.string())])
    batch = []

    def flush_batch():
        nonlocal batch
        if not batch:
            return
        part_name = f"part-{len(progress['parts']):06d}.parquet"
        part_path = destination / part_name
        temporary_part_path = destination / f"{part_name}.incomplete"
        if part_path.exists() or temporary_part_path.exists():
            raise FileExistsError(f"Refusing to overwrite existing Parquet shard: {part_path}")
        table = pa.Table.from_pydict(
            {"text": [record["text"] for record in batch], "language": [record["language"] for record in batch]},
            schema=schema,
        )
        print(f"Saving checkpoint to: {part_path}")
        pq.write_table(table, str(temporary_part_path), compression="zstd")
        temporary_part_path.replace(part_path)
        progress["parts"].append(part_name)
        progress["rows"] += len(batch)
        progress["actual_text_bytes"] += sum(len(record["text"].encode("utf-8")) for record in batch)
        progress["records_emitted"] += len(batch)
        progress["source_examples_consumed"] = batch[-1]["_source_examples_consumed"]
        checkpoint()
        batch = []

    for record in records:
        batch.append(record)
        if len(batch) >= shard_rows:
            flush_batch()
    flush_batch()

def save_streaming_parquet_target(target_id, destination, record_loader, metadata, shard_rows):
    if streaming_target_is_complete(target_id, destination):
        print(f"Checkpoint found; skipping {target_id}: {destination}")
        return manifest["targets"][target_id]

    temporary_destination = destination.with_name(f"{destination.name}.incomplete")
    if destination.exists():
        raise FileExistsError(f"Found an untracked completed folder for {target_id}: {destination}")

    progress_path = temporary_destination / "progress.json"
    if temporary_destination.exists():
        if not progress_path.is_file():
            raise FileExistsError(
                f"Found an incomplete folder without a resumable progress file: {temporary_destination}. "
                "Inspect it before retrying; this notebook will not overwrite local data."
            )
        progress = json.loads(progress_path.read_text(encoding="utf-8"))
        if progress.get("target_id") != target_id:
            raise ValueError(f"Progress file target mismatch in {progress_path}")
        for incomplete_part in temporary_destination.glob("*.parquet.incomplete"):
            incomplete_part.unlink()
        if not all(validate_parquet_shard(temporary_destination / part_name) for part_name in progress.get("parts", [])):
            raise ValueError(f"A completed checkpoint shard is invalid in {temporary_destination}")
        print(f"Resuming {target_id} from {progress['rows']:,} saved rows and {len(progress['parts'])} shards.")
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary_destination.mkdir(parents=True, exist_ok=False)
        progress = {
            **metadata, "target_id": target_id, "status": "running", "started_at_utc": utc_now(),
            "parts": [], "rows": 0, "actual_text_bytes": 0, "records_emitted": 0, "source_examples_consumed": 0,
        }
        atomic_write_json(progress_path, progress)

    manifest["targets"][target_id] = {
        "status": "running", "started_at_utc": progress["started_at_utc"],
        "destination": str(destination), "metadata": metadata,
        "progress": {key: progress[key] for key in ("rows", "actual_text_bytes", "records_emitted", "source_examples_consumed")},
    }
    save_manifest(manifest)

    def checkpoint():
        progress["updated_at_utc"] = utc_now()
        atomic_write_json(progress_path, progress)
        manifest["targets"][target_id]["progress"] = {
            key: progress[key] for key in ("rows", "actual_text_bytes", "records_emitted", "source_examples_consumed")
        }
        save_manifest(manifest)
        print(f"Checkpoint complete: {progress['rows']:,} rows across {len(progress['parts'])} shards.")

    try:
        write_parquet_shards(record_loader(progress), temporary_destination, progress, shard_rows, checkpoint)
        progress["status"] = "completed"
        progress["completed_at_utc"] = utc_now()
        atomic_write_json(progress_path, progress)
        atomic_write_json(temporary_destination / "metadata.json", progress)
        temporary_destination.replace(destination)
        manifest["targets"][target_id].update({
            "status": "completed", "completed_at_utc": progress["completed_at_utc"],
            "rows": progress["rows"], "actual_text_bytes": progress["actual_text_bytes"],
            "parts": len(progress["parts"]),
        })
        save_manifest(manifest)
        print(f"Saved {target_id} to: {destination}")
        return manifest["targets"][target_id]
    except Exception as error:
        manifest["targets"][target_id].update({
            "status": "failed", "failed_at_utc": utc_now(),
            "error": f"{type(error).__name__}: {error}",
        })
        save_manifest(manifest)
        raise


## 4. Download GLueCoS task datasets

Each requested NER, POS, and sentiment dataset is saved in its own local directory under `data/raw/gluecos/`.

In [5]:
GLUECOS_ROOT = RAW_DATA_ROOT / "gluecos"

for repository in download_config["gluecos"]:
    dataset_name = safe_path_component(repository)
    save_target(
        target_id=f"gluecos/{dataset_name}",
        destination=GLUECOS_ROOT / dataset_name,
        loader=lambda repository=repository: load_dataset(
            repository, cache_dir=str(HF_DATASETS_CACHE_ROOT), token=HF_TOKEN
        ),
        metadata={"repository": repository, "revision": "main"},
    )


README.md:   0%|          | 0.00/601 [00:00<?, ?B/s]

c:\Users\shenr\tokenization-research\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shenr\.cache\huggingface\hub\datasets--Huggmachas--GLuecos_NER_EN_HI. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Saving dataset to: C:\Users\shenr\tokenization-research\data\raw\gluecos\Huggmachas__GLuecos_NER_EN_HI.incomplete


Saving the dataset (0/1 shards):   0%|          | 0/306 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2458 [00:00<?, ? examples/s]

...

Saving dataset to: C:\Users\shenr\tokenization-research\data\raw\gluecos\Huggmachas__GLuecos_Sentiment_EN_ES.incomplete


Saving the dataset (0/1 shards):   0%|          | 0/184 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1524 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

Saved gluecos/Huggmachas__GLuecos_Sentiment_EN_ES to: C:\Users\shenr\tokenization-research\data\raw\gluecos\Huggmachas__GLuecos_Sentiment_EN_ES


## 5. Stream capped CulturaX language subsets

This section streams the first configured number of UTF-8 text bytes from each CulturaX language subset, without materializing the full source dataset or creating a second local Arrow copy. It saves independently finalized Parquet shards of the configured size, plus `progress.json`, under `data/raw/culturax/<language>.incomplete/`. The manifest and progress file update after every completed shard; after an interruption, rerunning this cell resumes from the next durable shard. The selection order is the dataset's source-stream order (no shuffle).

In [5]:
CULTURAX_REPOSITORY = download_config["sources"]["culturax"]

def iter_capped_culturax_records(language, progress):
    source = load_dataset(
        CULTURAX_REPOSITORY,
        language,
        streaming=True,
        cache_dir=str(HF_DATASETS_CACHE_ROOT),
        token=HF_TOKEN,
    )
    if hasattr(source, "keys"):
        if "train" not in source:
            raise KeyError(f"CulturaX/{language} has no train split: {list(source.keys())}")
        source = source["train"]

    source_examples_consumed = int(progress["source_examples_consumed"])
    if source_examples_consumed:
        source = source.skip(source_examples_consumed)
    bytes_written = int(progress["actual_text_bytes"])
    for example in source:
        source_examples_consumed += 1
        text = example.get("text")
        if not isinstance(text, str) or not text:
            continue
        text_bytes = len(text.encode("utf-8"))
        if bytes_written + text_bytes > MAX_TEXT_BYTES_PER_LANGUAGE:
            break
        bytes_written += text_bytes
        yield {
            "text": text,
            "language": language,
            "_source_examples_consumed": source_examples_consumed,
        }

for language in download_config["languages"]:
    metadata = {
        "repository": CULTURAX_REPOSITORY,
        "language": language,
        "revision": "main",
        "mode": "streaming_capped",
        "format": "parquet",
        "configured_text_byte_limit": MAX_TEXT_BYTES_PER_LANGUAGE,
        "parquet_shard_rows": PARQUET_SHARD_ROWS,
        "selection": "first valid records in source-stream order; no shuffle",
    }
    save_streaming_parquet_target(
        target_id=f"culturax/{language}",
        destination=RAW_DATA_ROOT / "culturax" / language,
        record_loader=lambda progress, language=language: iter_capped_culturax_records(language, progress),
        metadata=metadata,
        shard_rows=PARQUET_SHARD_ROWS,
    )


Resolving data files:   0%|          | 0/3072 [00:00<?, ?it/s]

Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000000.parquet
Checkpoint complete: 10,000 rows across 1 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000001.parquet
Checkpoint complete: 20,000 rows across 2 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000002.parquet
Checkpoint complete: 30,000 rows across 3 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000003.parquet
Checkpoint complete: 40,000 rows across 4 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000004.parquet
Checkpoint complete: 50,000 rows across 5 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000005.parquet
Checkpoint complete: 60,000 rows across 6 shards.
Saving checkpoint to: C:\Users\she

'peer closed connection without sending complete message body (received 65002201 bytes, expected 134918471)' thrown while requesting GET https://huggingface.co/datasets/uonlp/CulturaX/resolve/6a8734bc69fefcbb7735f4f9250f43e4cd7a442e/en/en_part_00000.parquet
Retrying in 1s [Retry 1/5].


Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000011.parquet
Checkpoint complete: 120,000 rows across 12 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000012.parquet
Checkpoint complete: 130,000 rows across 13 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000013.parquet
Checkpoint complete: 140,000 rows across 14 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000014.parquet
Checkpoint complete: 150,000 rows across 15 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000015.parquet
Checkpoint complete: 160,000 rows across 16 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\en.incomplete\part-000016.parquet
Checkpoint complete: 170,000 rows across 17 shards.
Saving checkpoint to: 

Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000000.parquet
Checkpoint complete: 10,000 rows across 1 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000001.parquet
Checkpoint complete: 20,000 rows across 2 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000002.parquet
Checkpoint complete: 30,000 rows across 3 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000003.parquet
Checkpoint complete: 40,000 rows across 4 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000004.parquet
Checkpoint complete: 50,000 rows across 5 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\es.incomplete\part-000005.parquet
Checkpoint complete: 60,000 rows across 6 shards.
Saving checkpoint to: C:\Users\she

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000000.parquet
Checkpoint complete: 10,000 rows across 1 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000001.parquet
Checkpoint complete: 20,000 rows across 2 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000002.parquet
Checkpoint complete: 30,000 rows across 3 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000003.parquet
Checkpoint complete: 40,000 rows across 4 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000004.parquet
Checkpoint complete: 50,000 rows across 5 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000005.parquet
Checkpoint complete: 60,000 rows across 6 shards.
Saving checkpoint to: C:\Users\she

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/uonlp/CulturaX/resolve/6a8734bc69fefcbb7735f4f9250f43e4cd7a442e/hi/hi_part_00000.parquet
Retrying in 1s [Retry 1/5].


Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000027.parquet
Checkpoint complete: 280,000 rows across 28 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000028.parquet
Checkpoint complete: 290,000 rows across 29 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000029.parquet
Checkpoint complete: 300,000 rows across 30 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000030.parquet
Checkpoint complete: 310,000 rows across 31 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000031.parquet
Checkpoint complete: 320,000 rows across 32 shards.
Saving checkpoint to: C:\Users\shenr\tokenization-research\data\raw\culturax\hi.incomplete\part-000032.parquet
Checkpoint complete: 330,000 rows across 33 shards.
Saving checkpoint to: 

## Download checkpoint summary

Rerun this summary after a failure or interruption. Completed targets will be reused on the next run. Inspect failed entries and any `.incomplete` folders before manually resolving them.

In [1]:
manifest = load_manifest()
status_counts = {}
for entry in manifest["targets"].values():
    status_counts[entry["status"]] = status_counts.get(entry["status"], 0) + 1

print(f"Manifest: {MANIFEST_PATH}")
print(f"Target status counts: {status_counts}")
for target_id, entry in sorted(manifest["targets"].items()):
    print(f"[{entry['status']}] {target_id} -> {entry['destination']}")


NameError: name 'load_manifest' is not defined